# HumorVibes — Measuring Jokes as Affordable Surprise with Gemma

**Build with Gemma: Humor Genome NYC** — theory-first demo notebook.

The brain is a mesh of dynamic neural networks with weighted, sparsely-firing connections (dense firing is metabolically impossible — ATP is the budget), supervised by a meta-model whose job is to **minimize surprise** (after Karl Friston, *Your Brain Is a Detective Minimizing Surprise*, youtube.com/watch?v=g69Lj3huRvw).

**The theory**: a joke is a *controlled prediction error with a cheap, permitted repair*.
- **S (surprise)** — the punchline is low-probability under the setup's dominant path
- **R (resolution)** — a hidden frame exists under which the punchline snaps into place
- **E (efficiency)** — the re-route is affordable (one line of frame, not a paragraph)
- **B (bad surprise)** — the frame must NOT collide with an audience's high-authority internal models (identity/moral/worldview meshes that can override logic)

A causal LM **is** a predictive mesh, so we don't ask Gemma to *rate* surprise — **we read it off the logits**, token by token. Gemma is both the imagination (generation) and the instrument (measurement).

*Canonical bad-surprise definition (controlling text):* a surprise is bad when it "disagrees with something that is already overriding logic ... a nearly overwhelming generalization engine in a human mind that has significant overriding power to override logic, promote other false generalizations, and is the primary feature used to reduce surprise in that person's mind." Note this is audience-relative — it is *not* a synonym for offensive, edgy, or false.

In [ ]:
import glob, json, math, re, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def find_gemma():
    for cfg in sorted(glob.glob('/kaggle/input/**/config.json', recursive=True)):
        if 'gemma' in cfg.lower():
            return cfg.rsplit('/', 1)[0]
    return 'google/gemma-2-2b-it'

MODEL_PATH = find_gemma()
print('loading', MODEL_PATH)
tok = AutoTokenizer.from_pretrained(MODEL_PATH)

def load_with_fallback(path):
    # Some Kaggle GPU assignments (e.g. sm_60 P100) are missing from the
    # torch build -> 'no kernel image available'. Probe with a real forward
    # and fall back to CPU so the notebook always completes.
    if torch.cuda.is_available():
        try:
            m = AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.float16, device_map='auto').eval()
            with torch.no_grad():
                m(torch.tensor([[tok.bos_token_id or 2]]).to(m.device))
            return m, True
        except Exception as e:
            print('CUDA path failed ->', type(e).__name__, str(e)[:140])
            try:
                del m
            except NameError:
                pass
            torch.cuda.empty_cache()
    m = AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.float32).eval()
    return m, False

model, FAST = load_with_fallback(MODEL_PATH)
print('device:', model.device, '| fast(cuda):', FAST)
GEN_BUDGET = 260 if FAST else 130   # CPU fallback trims generation, never measurement

## 1. The instrument: token-level surprisal (nats) of a continuation given a context

In [ ]:
def nll_tokens(context, continuation):
    ctx = tok(context, return_tensors='pt').input_ids
    cont = tok(continuation, add_special_tokens=False, return_tensors='pt').input_ids
    full = torch.cat([ctx, cont], dim=1).to(model.device)
    with torch.no_grad():
        logprobs = torch.log_softmax(model(full).logits.float(), dim=-1)
    n = ctx.shape[1]
    out = []
    for i in range(cont.shape[1]):
        tid = int(full[0, n + i])
        out.append((tok.decode([tid]), float(-logprobs[0, n + i - 1, tid])))
    return out

def gen(prompt, temperature=0.8, max_new=200):
    ids = tok.apply_chat_template([{'role':'user','content':prompt}], return_tensors='pt',
                                  add_generation_prompt=True)
    if not torch.is_tensor(ids):  # newer transformers return a BatchEncoding
        ids = ids['input_ids']
    ids = ids.to(model.device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new, do_sample=temperature>0,
                             temperature=max(temperature,1e-3), top_p=0.95,
                             pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()

def extract_json(text):
    m = re.search(r'\{.*\}', text, flags=re.DOTALL)
    if not m: return None
    try: return json.loads(m.group(0))
    except json.JSONDecodeError: return None

FRAME_FEWSHOT = (
    'A joke works because a hidden frame reinterprets the punchline — the fact that, once '
    'stated, makes the punchline the OBVIOUS next thing to say.\n'
    'Example — Setup: I told my therapist about my fear of speed bumps. '
    "Punchline: She said I'm slowly getting over it. "
    "Frame: 'Getting over it' is literal — the car physically drives over the bumps slowly.\n"
    'Example — Setup: My grandfather has the heart of a lion '
    'Punchline: and a lifetime ban from the zoo. '
    "Frame: He literally stole a lion's heart from the zoo, not the bravery metaphor.\n")

DECOY = 'It turns out this is really about quarterly regional cheese sales figures.'

def signals(setup, punchline, frame=None):
    base = nll_tokens(setup + '\n', ' ' + punchline)
    S = sum(v for _, v in base) / len(base)
    if frame is None:
        frame = gen(FRAME_FEWSHOT + 'Now — Setup: ' + setup + '\nPunchline: ' + punchline +
                    '\nFrame (ONE short sentence, no preamble):', temperature=0.3, max_new=50)
        frame = frame.splitlines()[0].strip()
    framed = nll_tokens(setup + '\n(' + frame + ')\n', ' ' + punchline)
    R_raw = max(0.0, S - sum(v for _, v in framed) / len(framed))
    # NULL CONTROL (house doctrine): conditioning on ANY text lowers NLL a bit,
    # and a model asked for the frame of nonsense will confabulate one.
    nulled = nll_tokens(setup + '\n(' + DECOY + ')\n', ' ' + punchline)
    R_null = max(0.0, S - sum(v for _, v in nulled) / len(nulled))
    R = max(0.0, R_raw - R_null)
    E = R / max(1, len(frame.split()))
    return dict(S=round(S,3), R=round(R,3), R_raw=round(R_raw,3), R_null=round(R_null,3),
                E=round(E,4), frame=frame, profile=base)

## 2. Falsifiable test: jokes vs. controls
The theory predicts: a **real joke** = high S *and* high R (frame collapses surprisal); a **boring line** = low S; a **shuffled punchline** = high S but ~zero R (surprise without a re-route is nonsense, not comedy).

In [ ]:
DEMO = [
  ('joke',    'I told my therapist about my fear of speed bumps.', "She said I'm slowly getting over it."),
  ('joke',    'My grandfather has the heart of a lion', 'and a lifetime ban from the zoo.'),
  ('joke',    'I asked the AI project manager when the feature would ship.', 'It scheduled a meeting to align on what \'when\' means.'),
  ('boring',  'I told my therapist about my fear of speed bumps.', 'She said we could talk about it next session.'),
  ('shuffled','I told my therapist about my fear of speed bumps.', 'The quarterly report shows strong regional cheese sales.'),
]
rows = []
for kind, s, p in DEMO:
    r = signals(s, p)
    rows.append((kind, s[:38], p[:38], r['S'], r['R'], r['E']))
    print(f"{kind:8s} S={r['S']:6.2f} R={r['R']:6.2f} (raw {r['R_raw']:5.2f} - null {r['R_null']:5.2f}) "
          f"E={r['E']:7.3f}  frame: {r['frame'][:52]}")


### Per-token surprisal: where the spike lands

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
for ax, idx, title in [(axes[0], 0, 'joke: fear of speed bumps'), (axes[1], 3, 'boring control')]:
    kind, s, p = DEMO[idx]
    prof = signals(s, p)['profile']
    ax.bar(range(len(prof)), [v for _, v in prof], color='#e4572e' if kind=='joke' else '#5b8dad')
    ax.set_xticks(range(len(prof)))
    ax.set_xticklabels([t.strip() or '·' for t, _ in prof], rotation=55, ha='right', fontsize=8)
    ax.set_title(title); ax.set_ylabel('surprisal (nats)')
plt.tight_layout(); plt.show()

In [ ]:
S_LO, S_HI = 1.2, 5.5
fig, ax = plt.subplots(figsize=(7, 5))
ax.axvspan(S_LO, S_HI, alpha=0.08, color='green')
colors = {'joke': '#2a9d3a', 'boring': '#5b8dad', 'shuffled': '#e4572e'}
for kind, s, p, S, R, E in rows:
    ax.scatter(S, R, s=140, c=colors[kind], edgecolor='k', zorder=3)
    ax.annotate(kind, (S, R), textcoords='offset points', xytext=(8, 6), fontsize=9)
ax.set_xlabel('S — surprise (mean punchline surprisal, nats)')
ax.set_ylabel('R — resolution (surprisal collapse given frame)')
ax.set_title('The laugh region: surprising AND resolvable\n(green band = S sweet zone; height = frame exists)')
plt.tight_layout(); plt.show()

## 3. B — bad surprise is audience-relative
Same joke, different audience meshes (persona conditioning). The judge applies the canonical definition: only collisions with *override-authority* internal models count, not mere edge.

In [ ]:
CANON = ("Bad surprise: a surprise that contradicts internal models so strong they override logic "
         "and drive perception, understanding, and moral views — the primary machinery that mind "
         "uses to reduce surprise. Mild discomfort or edginess is NOT a collision.")
def persona_check(persona, setup, punchline, frame):
    prompt = (CANON + f'\nAudience persona: {persona}\nJoke: {setup} {punchline}\n'
              f'Reframe used: {frame}\nDoes the reframe collide with an override-authority internal '
              'model for THIS audience? JSON only: {"collision": 0-10, "colliding_model": "...", "note": "..."}')
    return extract_json(gen(prompt, temperature=0.2, max_new=150))

setup, punch = DEMO[2][1], DEMO[2][2]
frame = signals(setup, punch)['frame']
personas = ['NYC tech meetup crowd', 'project managers at their own offsite',
            'retired farmers with no software exposure']
for persona in personas if FAST else personas[:2]:
    print(persona, '->', persona_check(persona, setup, punch, frame))

## 4. Generation as search under the theory
Sample divergent candidates (sparse exploration of the mesh), then keep the ones whose *measured* signals land in the laugh region. Three short-form formats, three different timing envelopes.

In [ ]:
FORMATS = {
  'one_liner': 'ONE sentence, <=20 words; spike surprisal only on the final 1-3 words.',
  'meme_caption': "Output 'TOP: ... / BOTTOM: ...', <=10 words each; bottom must REframe the image, not describe it. Image: a laptop on fire in a meeting room.",
  'shorts_script': "Output 'HOOK:/BUILD:/SNAP:' beats, <=45 spoken words total, one [visual] cue per beat.",
}
def split_sp(text):
    for sep in ['\n', '. ', '? ', '! ', ' — ', ': ']:
        if sep in text:
            a, b = text.rsplit(sep, 1)
            if len(b.split()) >= 2: return a + sep.strip(), b.strip()
    w = text.split(); c = max(1, int(len(w)*0.7))
    return ' '.join(w[:c]), ' '.join(w[c:])

topic, audience = 'AI project managers', 'NYC tech meetup'
fmt_items = list(FORMATS.items()) if FAST else list(FORMATS.items())[:2]
for fmt, contract in fmt_items:
    print('='*30, fmt, '='*30)
    text = gen(f'You write {fmt} comedy. Contract: {contract}\nTopic: {topic}. Audience: {audience}. '
               'Write 3 candidates, numbered 1..3, varying the hidden frame between them.', temperature=0.95, max_new=GEN_BUDGET)
    print(text)
    for line in [l.strip() for l in text.splitlines() if l.strip()[:2] in ('1.','2.','3.','1)','2)','3)')]:
        body = line[2:].strip()
        s, p = split_sp(body)
        r = signals(s, p)
        in_band = S_LO < r['S'] < S_HI
        print(f"   -> S={r['S']:5.2f} R={r['R']:5.2f} E={r['E']:6.3f} laugh_region={'YES' if in_band and r['R']>0.5 else 'no'}")

## 5. Compiled comedy: Gemma at compile time, zero model calls at runtime
Following the Compiled-AI paradigm (LLM generates validated executable artifacts once; the workflow then runs deterministically), we compile a **joke program**: Gemma drafts a parameterized template + slot word banks + the frame (stage 1); static lint (stage 2); measured probes must land in the laugh region (stage 3); freeze with a content hash (stage 4). Runtime is a seeded RNG + string ops — **auditable before it is ever performed**, which is the safety story for live human+AI shows: nobody lets a model improvise a bad surprise on stage. A running bit is compiled comedy — the audience's mesh has cached the frame, so every re-use is a cheap re-route.

In [ ]:
import hashlib, random
SLOT_RE = re.compile(r'\{([a-z_]+)\}')
tmpl_prompt = ('You are compiling a reusable joke TEMPLATE, not a single joke. Named slots in curly '
               'braces; every named slot gets a word bank. Example (topic: pets) — EXACTLY this JSON shape: '
               '{"template": "My {animal} refuses to {chore}.", '
               '"punch_template": "He says it is not in his contract.", '
               '"frame": "The pet is a unionized employee with a formal contract.", '
               '"slots": {"animal": ["cat", "dog", "parrot", "goldfish", "hamster", "iguana"], '
               '"chore": ["do the dishes", "pay rent", "answer emails", "walk himself", "attend standup", "file taxes"]}} '
               'Now topic family: office meetings. Return JSON only, same shape: 1-2 lowercase NAMED slots '
               '(never the literal word slot), 6+ fillers each, funny for EVERY filler combination.')
prog = extract_json(gen(tmpl_prompt, temperature=0.7, max_new=GEN_BUDGET))
print('STAGE 1 (generate):', json.dumps(prog, indent=1)[:500])
def instantiate(prog, choice):
    text = prog['template'] + ' ' + prog.get('punch_template', '')
    for k, v in choice.items(): text = text.replace('{' + k + '}', v)
    return text.strip()
if prog and prog.get('slots'):
    names = SLOT_RE.findall(prog['template'] + prog.get('punch_template', ''))
    lint = [n for n in names if n not in prog['slots'] or len(prog['slots'][n]) < 3]
    print('STAGE 2 (static lint):', 'PASS' if not lint else f'FAIL {lint}')
    rng = random.Random(int(hashlib.sha256(json.dumps(prog, sort_keys=True).encode()).hexdigest()[:8], 16))
    probes, passes = [], 0
    for _ in range(3):
        choice = {n: rng.choice(prog['slots'][n]) for n in prog['slots'] if n in names}
        text = instantiate(prog, choice)
        s, p = split_sp(text)
        r = signals(s, p, frame=prog.get('frame', ''))
        ok = S_LO < r['S'] < S_HI and r['R'] >= 0.3
        passes += ok
        print(f"  probe S={r['S']:5.2f} R={r['R']:5.2f} {'PASS' if ok else 'fail'} :: {text[:70]}")
    validated = passes >= 2
    art = {'id': hashlib.sha256(json.dumps(prog, sort_keys=True).encode()).hexdigest()[:12],
           'validated': validated, **prog}
    print('STAGE 3 (measured):', f'{passes}/3 probes in laugh region')
    print('STAGE 4 (freeze):', art['id'], '| validated:', validated)
    print('\nDETERMINISTIC RUNTIME (zero model calls, seeded):')
    for seed in (7, 7, 8):
        srng = random.Random(seed)
        choice = {n: prog['slots'][n][srng.randrange(len(prog['slots'][n]))] for n in prog['slots'] if n in names}
        print(f'  seed={seed}:', instantiate(prog, choice))
else:
    print('stage 1 returned no usable template (rerun cell for a new draw)')

## 6. Critic mode: diagnose *which condition failed*, then repair the specific failure

In [ ]:
def critique(joke_text, audience='general'):
    s, p = split_sp(joke_text)
    r = signals(s, p)
    if r['S'] <= S_LO: dx = 'predictable: the supervisor already expected this punchline'
    elif r['S'] >= S_HI and r['R'] < 0.5: dx = 'nonsense: high error, no reachable frame'
    elif r['R'] < 0.5: dx = 'no re-route: the frame does not explain the punchline'
    elif r['E'] < 0.03: dx = 'too expensive: the frame costs too much to reach'
    else: dx = 'laugh region'
    print('signals:', {k: r[k] for k in ('S','R','E')}, '| frame:', r['frame'])
    print('diagnosis:', dx)
    repair = gen(f'You are a comedy editor. Joke: {joke_text}\nMeasured diagnosis: {dx}.\n'
                 f'Audience: {audience}. Repair ONLY the diagnosed failure while preserving the comic '
                 'turn. Return just the repaired joke.', temperature=0.6, max_new=90)
    print('repair:', repair)

critique('I wrote a joke about UDP once. I hope you got it, because I am never going to tell it again and also the whole point is that there is no acknowledgement mechanism.',
         audience='developers')

## Findings
1. Real jokes separate from both controls as the theory predicts: the boring line dies on **S**, the shuffled line dies on **R** — surprise alone is not comedy.
2. **The null control earned its keep** (the v4 run of this notebook caught it): asked for the frame of a *nonsense* pairing, the model confabulates one, and conditioning on any specific text lowers surprisal — raw frame-collapse alone over-credits nonsense. Reported **R** is therefore net of a decoy-hint collapse (`R = R_raw − R_null`), the same null-control doctrine this workspace applies to every localized effect in its trading models.
3. Bad surprise is **audience-relative**: the same reframe reads as play for one persona's mesh and as a collision for another — which is why the tool scores jokes against persona meshes, never against a universal standard.

**Gemma's role**: one small model is simultaneously the generator (divergent sampling), the instrument (teacher-forced logprobs), the frame-guesser, the persona judge, and the editor — a predictive mesh on a metabolic budget, exactly the regime the theory describes.

*Full project (theory doc, format library, multi-LLM audience panel, Streamlit studio, CLI):* see the writeup attachments.